In [1]:
import pandas as pd

data = pd.read_csv("trainingData.csv")

print("行数和列数：", data.shape)
data[["WAP001", "WAP002", "WAP003", "BUILDINGID", "FLOOR"]].head()

行数和列数： (19937, 529)


,WAP001,WAP002,WAP003,BUILDINGID,FLOOR
0,100,100,100,1,2
1,100,100,100,1,2
2,100,100,100,1,2
3,100,100,100,1,2
4,100,100,100,0,0


In [2]:
# 找到所有表示 Wi-Fi 热点的列
wifi_columns = [col for col in data.columns if col.startswith("WAP")]

# 取第一条扫描，只保留实际检测到的信号
first_scan = data.loc[0, wifi_columns]
detected = first_scan[first_scan != 100]

print("数据集中的热点总数：", len(wifi_columns))
print("第一条扫描实际检测到的热点数：", len(detected))
print("这条记录的正确答案：")
print(data.loc[0, ["BUILDINGID", "FLOOR"]])

detected.head(10)

数据集中的热点总数： 520
第一条扫描实际检测到的热点数： 17
这条记录的正确答案：
BUILDINGID    1.0
FLOOR         2.0
Name: 0, dtype: float64


WAP090   -67.0
WAP091   -68.0
WAP103   -68.0
WAP104   -68.0
WAP105   -78.0
WAP106   -79.0
WAP125   -92.0
WAP126   -95.0
WAP166   -91.0
WAP167   -89.0
Name: 0, dtype: float64

In [3]:
# X 是模型的输入：只取 Wi-Fi 信号列
X = data[wifi_columns].copy()

# y 是正确答案：把楼栋和楼层合成一个标签
y = data["BUILDINGID"].astype(str) + "_" + data["FLOOR"].astype(str)

print("输入表的形状：", X.shape)
print("答案的数量：", len(y))
print("不同的楼栋＋楼层组合数：", y.nunique())
print("前 5 条记录的答案：")
print(y.head())

输入表的形状： (19937, 520)
答案的数量： 19937
不同的楼栋＋楼层组合数： 13
前 5 条记录的答案：
0    1_2
1    1_2
2    1_2
3    1_2
4    0_0
dtype: object


In [4]:
from sklearn.model_selection import StratifiedGroupKFold

# 同一栋楼、同一层、同一个坐标的记录归到同一组
location_group = data.groupby(
    ["BUILDINGID", "FLOOR", "LONGITUDE", "LATITUDE"]
).ngroup()

# 计划分成五份，取其中一份验证，其余四份训练
splitter = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=42
)
train_idx, val_idx = next(splitter.split(X, y, groups=location_group))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

shared_locations = set(location_group.iloc[train_idx]) & set(location_group.iloc[val_idx])

print("训练记录数：", len(X_train))
print("验证记录数：", len(X_val))
print("训练和验证的类别数：", y_train.nunique(), y_val.nunique())
print("两组共享的采集位置数：", len(shared_locations))

训练记录数： 15950
验证记录数： 3987
训练和验证的类别数： 13 13
两组共享的采集位置数： 0


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 用一个低值编码“未检测到”，原始数据里的 100 保持原样
X_train_model = X_train.replace(100, -110)
X_val_model = X_val.replace(100, -110)

# 标准化参数只从训练集计算，再用同一套参数处理验证集
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train_model, y_train)

pred_original = model.predict(X_val_model)
acc_original = accuracy_score(y_val, pred_original)

print(f"原始验证记录的定位准确率：{acc_original:.2%}")
print(f"猜对了 {(pred_original == y_val).sum()} / {len(y_val)} 条")

原始验证记录的定位准确率：96.97%
猜对了 3866 / 3987 条


In [6]:
import numpy as np

# 从训练集统计每个热点被检测到多少次
detection_counts = X_train.ne(100).sum()
available_hotspots = detection_counts[detection_counts > 0].index
top5_hotspots = detection_counts.nlargest(5).index.tolist()

def evaluate_outage(hotspots):
    # 先数有多少条扫描原本能看到至少一个将被停用的热点
    affected = X_val[hotspots].ne(100).any(axis=1).sum()

    # 在验证集的副本上模拟故障，再沿用之前的输入编码
    X_outage = X_val.copy()
    X_outage.loc[:, hotspots] = 100
    predictions = model.predict(X_outage.replace(100, -110))

    return accuracy_score(y_val, predictions), affected

# 随机抽 20 组，每组停用 5 个热点，最后取平均
rng = np.random.default_rng(42)
random_results = []

for _ in range(20):
    hotspots = rng.choice(available_hotspots, size=5, replace=False).tolist()
    random_results.append(evaluate_outage(hotspots))

acc_random, affected_random = np.mean(random_results, axis=0)
acc_top5, affected_top5 = evaluate_outage(top5_hotspots)

print("最常被检测到的 5 个热点：", top5_hotspots)

pd.DataFrame({
    "验证条件": ["原始记录", "随机停用 5 个（20 组平均）", "停用 5 个常见热点"],
    "定位准确率": [f"{a:.2%}" for a in [acc_original, acc_random, acc_top5]],
    "比原始下降（百分点）": [
        f"{100 * (acc_original - a):.2f}"
        for a in [acc_original, acc_random, acc_top5]
    ],
    "实际少了信号的记录数": [0, round(affected_random, 1), affected_top5],
})

最常被检测到的 5 个热点： ['WAP087', 'WAP496', 'WAP502', 'WAP517', 'WAP078']


,验证条件,定位准确率,比原始下降（百分点）,实际少了信号的记录数
0,原始记录,96.97%,0.00,0.0
1,随机停用 5 个（20 组平均）,96.73%,0.23,695.0
2,停用 5 个常见热点,96.51%,0.45,1576.0


In [7]:
# 找出原本能看到至少一个常见热点的扫描
affected = X_val[top5_hotspots].ne(100).any(axis=1)

# 模拟常见热点故障，用之前训练好的模型预测
X_top5 = X_val.copy()
X_top5.loc[:, top5_hotspots] = 100
pred_top5 = model.predict(X_top5.replace(100, -110))

was_correct = pred_original == y_val.to_numpy()
now_correct = pred_top5 == y_val.to_numpy()

print("受影响的扫描数：", affected.sum())
print("从猜对变成猜错：", (affected & was_correct & ~now_correct).sum())
print("从猜错变成猜对：", (affected & ~was_correct & now_correct).sum())

受影响的扫描数： 1576
从猜对变成猜错： 25
从猜错变成猜对： 7


In [8]:
# 找到第一条原来猜对、故障后猜错的记录
new_errors = affected.to_numpy() & was_correct & ~now_correct
i = np.flatnonzero(new_errors)[0]

# 查看这条记录里，哪些原本可见的热点被停用了
signals = X_val.iloc[i][top5_hotspots]
lost_signals = signals[signals != 100]

print("真实位置：", y_val.iloc[i])
print("故障前预测：", pred_original[i])
print("故障后预测：", pred_top5[i])
print("实际丢失的热点数：", len(lost_signals))

lost_signals.rename("故障前的信号强度（dBm）").to_frame()

真实位置： 2_3
故障前预测： 2_3
故障后预测： 2_4
实际丢失的热点数： 4


,故障前的信号强度（dBm）
WAP087,-85
WAP496,-77
WAP502,-76
WAP517,-87


In [9]:
# 找出每列对应的热点，并标记哪几列是常见热点
values = X_val.to_numpy()
is_top5 = X_val.columns.isin(top5_hotspots)
visible = values != 100

# 每条扫描在常见热点故障下实际丢失的信号数
lost_count = visible[:, is_top5].sum(axis=1)

# 只比较确实丢了信号，而且有足够其他信号可删的扫描
other_count = visible[:, ~is_top5].sum(axis=1)
rows = np.flatnonzero((lost_count > 0) & (other_count >= lost_count))

X_compare = X_val.iloc[rows]
y_compare = y_val.iloc[rows]

# 在同一批扫描上，重复 20 轮随机删点
rng = np.random.default_rng(42)
random_scores = []

for _ in range(20):
    masked = X_compare.to_numpy(copy=True)
    for j, row in enumerate(rows):
        candidates = np.flatnonzero(visible[row] & ~is_top5)
        removed = rng.choice(candidates, size=lost_count[row], replace=False)
        masked[j, removed] = 100

    X_random_matched = pd.DataFrame(masked, columns=X_compare.columns)
    pred = model.predict(X_random_matched.replace(100, -110))
    random_scores.append(accuracy_score(y_compare, pred))

print("可比较的扫描数：", len(rows))
print(f"原始记录：{accuracy_score(y_compare, pred_original[rows]):.2%}")
print(f"随机删同样多的其他信号（20 轮平均）：{np.mean(random_scores):.2%}")
print(f"删除常见热点信号：{accuracy_score(y_compare, pred_top5[rows]):.2%}")

可比较的扫描数： 1576
原始记录：98.03%
随机删同样多的其他信号（20 轮平均）：93.91%
删除常见热点信号：96.89%


In [10]:
results = []

def add_result(condition, k, accuracy, affected_count):
    results.append({
        "故障条件": condition,
        "停用热点数": k,
        "准确率（%）": 100 * accuracy,
        "比原始下降（百分点）": 100 * (acc_original - accuracy),
        "实际丢失信号的扫描数": affected_count,
    })

add_result("原始记录", 0, acc_original, 0)

for k in [5, 10, 20]:
    # 随机停用：每档抽 20 组，每组停用 k 个热点
    rng = np.random.default_rng(42)
    random_runs = []

    for _ in range(20):
        hotspots = rng.choice(available_hotspots, size=k, replace=False).tolist()
        random_runs.append(evaluate_outage(hotspots))

    avg_accuracy, avg_affected = np.mean(random_runs, axis=0)
    add_result("随机停用（20 组平均）", k, avg_accuracy, avg_affected)

    # 常见热点停用：按训练集里的检测次数挑前 k 个
    common_hotspots = detection_counts.nlargest(k).index.tolist()
    accuracy, affected_count = evaluate_outage(common_hotspots)
    add_result("停用常见热点", k, accuracy, affected_count)

outage_table = pd.DataFrame(results)
outage_table.round(2)

,故障条件,停用热点数,准确率（%）,比原始下降（百分点）,实际丢失信号的扫描数
0,原始记录,0,96.97,0.00,0.00
1,随机停用（20 组平均）,5,96.73,0.23,695.05
2,停用常见热点,5,96.51,0.45,1576.00
3,随机停用（20 组平均）,10,96.31,0.66,1426.75
4,停用常见热点,10,94.48,2.48,2144.00
5,随机停用（20 组平均）,20,95.39,1.57,2232.70
6,停用常见热点,20,90.12,6.85,2371.00
